In [5]:
from langchain.agents.middleware import ModelRequest, dynamic_prompt


@dynamic_prompt
def persionalized_prompt(req: ModelRequest) -> str:
    runtime = getattr(req, "runtime", None)
    user_id = "访客"
    if runtime and getattr(runtime, "context", None):
        user_id = runtime.context.get("user_id", "访客")
    return f"你是一名贴心的中文助手，正在为用户 <{user_id}> 提供帮助，回答时要简洁，自然。"


from typing import Any

from langchain.agents.middleware import AgentState, before_agent
from langgraph.runtime import Runtime


@before_agent
def log_before_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"[Before_agent] 本次会话开始， 已有消息数: {len(state.get('messages'))}")
    return None


from langchain.agents.middleware import after_agent


@after_agent
def log_after_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"[After_agent] 本次会话结束， 最终消息数: {len(state.get('messages'))}")
    return None


from langchain.agents.middleware import before_model


@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"[Before_model] 准备进行模型调用，当前消息数: {len(state.get('messages'))}")
    return None


from langchain.agents.middleware import after_model


@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"[After_model] 模型调用结束，当前消息数为: {len(state.get('messages'))}")
    return None


from langchain.messages import AIMessage


@after_model
def validate_output(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """简单的输出校验: 若模型输出包含 'BLOCKED_CN', 则改写消息并跳转到 end"""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and "BLOCKED_CN" in (last.content or ""):
        print("[After_model] 触发安全规则: 检测到 BLOCKED_CN, 跳转到end")
        return {
            "messages": [AIMessage("该请求触发了安全校验，无法继续！")],
            "jump_tp": "end",
        }


import time
from typing import Callable

from langchain.agents.middleware import ModelResponse, wrap_model_call


@wrap_model_call
def retry_and_timing(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    max_retries = 2
    start = time.time()
    try:
        for i in range(max_retries + 1):
            try:
                return handler(request)
            except Exception as e:
                if i == max_retries:
                    raise
                backoff = 0.1 * 2**i
                print(f"[wrap_model_call] 调用失败， 将在 {backoff:.2f}s 后重试 {e}")
                time.sleep(backoff)
    finally:
        cost = time.time() - start
        print(f"[wrap_model_call] 本次调用耗时 {cost: .3f}s")


import os

import dotenv
from langchain.chat_models import init_chat_model

dotenv.load_dotenv()
llm = init_chat_model(
    model="deepseek-chat",
    base_url="https://api.deepseek.com/v1",
    api_key=os.getenv("DEEPSEEK_API"),
    temperature=0.7,
    model_provider="openai",
)

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    middleware=[
        persionalized_prompt,
        log_before_agent,
        log_after_agent,
        log_before_model,
        log_after_model,
        retry_and_timing,
        validate_output,
    ],
)

from langchain.messages import HumanMessage

result = agent.invoke(
    {"messages": [HumanMessage("用一句话解释LangChain是什么?")]}, config={"context": {"user_id": "Alice"}}
)
print(result)

result = agent.invoke({"messages": [HumanMessage("请只回复: BLOCKED_CN")]}, config={"context": {"user_id": "Bob"}})
print(result)


[Before_agent] 本次会话开始， 已有消息数: 1
[Before_model] 准备进行模型调用，当前消息数: 1
[wrap_model_call] 本次调用耗时  1.105s
[After_model] 模型调用结束，当前消息数为: 2
[After_agent] 本次会话结束， 最终消息数: 2
{'messages': [HumanMessage(content='用一句话解释LangChain是什么?', additional_kwargs={}, response_metadata={}, id='fcd94403-eb06-45d0-9e1d-d513a6418fa0'), AIMessage(content='LangChain是一个用于构建大语言模型应用的开发框架，帮助开发者将LLM与其他工具和数据源连接起来。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 34, 'total_tokens': 58, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 34}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '1a42d5b6-2ed4-4742-ab2e-30b35d2c7659', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f22bd-54c9-7cd1-a608-dc43fb472e75-0', tool_calls=[], invalid_tool_calls=[], us